# Séance 9 · Exercices — Modéliser : ton premier modèle de machine learning · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

**Niveau de la séance : ⭐⭐ Intermédiaire** · chaque exercice porte son propre niveau (⭐ Débutant · ⭐⭐ Intermédiaire · ⭐⭐⭐ Avancé).

Comment travailler :
1. Lis l'énoncé, puis code dans la cellule « À toi » (les variables à remplir sont déjà nommées).
2. Lance la cellule de vérification : ✅ c'est bon, ❌ on réessaie (l'indice est là pour ça).
3. Ouvre la solution **seulement après avoir essayé**.

Ce notebook tourne dans **Google Colab** : rien à installer. Pour exécuter une cellule : clique dedans puis `Maj + Entrée`.

La recette de la leçon, sur des données qu'elle n'a pas vues : **392 voitures** des années 1970-1982 (deviner d'où elles viennent, puis leur consommation), **53 940 diamants** (retrouver les rares pierres de la pire pureté) et, pour le défi, les **244 additions** d'un restaurant.

Fais les exercices dans l'ordre : de l'exercice 2 à l'exercice 9, chacun réutilise les variables du précédent.

## Quiz d'ouverture (2 min)

Réponds dans ta tête, puis déplie la réponse. Ce sont les questions qu'on se pose à voix haute au début de la séance.

**1. Pourquoi cache-t-on une partie des données avant d'entraîner ?**
- a. Pour que ça aille plus vite
- b. Pour vérifier le modèle sur des exemples qu'il n'a jamais vus
- c. Parce qu'il y en a trop
- d. Pour économiser de la mémoire

<details><summary>Réponse</summary>

**b. Pour vérifier le modèle sur des exemples qu'il n'a jamais vus** — Un élève qui a les réponses pendant le contrôle aura 20/20 sans avoir compris. Même chose pour un modèle.

</details>

**2. Prédire le prix d'un jeu vidéo, c'est de la…**
- a. Classification
- b. Régression
- c. Génération
- d. Recherche

<details><summary>Réponse</summary>

**b. Régression** — Un prix est un nombre, donc régression. Deviner le genre du jeu serait de la classification.

</details>

**3. Un arbre très profond fait 100 % sur les données d'entraînement et 70 % sur le test. C'est…**
- a. Un excellent modèle
- b. Du sur-apprentissage
- c. Un bug de scikit-learn
- d. Normal, le test est plus dur

<details><summary>Réponse</summary>

**b. Du sur-apprentissage** — Il a appris ses exemples par cœur. Un arbre moins profond fera un peu moins bien en entraînement mais mieux sur les nouveaux cas.

</details>

**4. On veut deviner d'où vient une voiture (Europe, Japon, États-Unis) à partir de son poids et de sa consommation. Que désigne y ?**
- a. Le poids et la consommation
- b. La colonne origine, ce qu'on doit deviner
- c. Le nombre de voitures
- d. L'exactitude du modèle

<details><summary>Réponse</summary>

**b. La colonne origine, ce qu'on doit deviner** — X, ce que la machine voit : les mesures. y, ce qu'elle doit deviner : l'origine.

</details>

## Préparation

Trois fichiers publics de la bibliothèque seaborn, chargés depuis Internet (si le site de seaborn ne répond pas, la copie rangée dans le dossier `donnees/` du cours prend le relais) :
- `voitures` : 392 modèles de voitures vendus aux États-Unis entre 1970 et 1982 (les 6 lignes incomplètes sont retirées). `mpg` = miles par gallon, plus c'est haut moins la voiture consomme ; `displacement` = cylindrée (en pouces cubes) ; `horsepower` = chevaux ; `weight` = poids (en livres) ; `origin` = `usa`, `japan` ou `europe`.
- `diamants` : 53 940 diamants, leur poids en carats, leurs proportions (`depth`, `table`), leur prix en dollars et leur pureté (`clarity`, de `IF`, sans défaut, à `I1`, la pire).
- `additions` : 244 additions d'un restaurant (montant, pourboire, nombre de convives, déjeuner ou dîner).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error

# le dépôt seaborn-data, puis la copie du cours (dans le dépôt cloné, puis en ligne), qui ne dépend pas de lui
SOURCES = ["https://raw.githubusercontent.com/mwaskom/seaborn-data/master/",
           "../../donnees/",
           "https://raw.githubusercontent.com/EricBlanvillain/cours-data-science/main/donnees/"]

def charger(nom):
    """Charge un fichier seaborn, depuis la première source qui répond."""
    for source in SOURCES:
        try:
            return pd.read_csv(source + nom + ".csv")
        except Exception as erreur:
            print("Pas de réponse de", source, ":", erreur)
    print("Pas de réseau ? Impossible de charger", nom)

voitures = charger("mpg").dropna().reset_index(drop=True)   # un modèle n'aime pas les cases vides
diamants = charger("diamonds")
additions = charger("tips")
print("Voitures :", voitures.shape, "· diamants :", diamants.shape, "· additions :", additions.shape)

resultats = {}

def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais lever d'exception (condition = un booléen, ou une fonction sans argument)."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception:
        ok = False
    resultats[nom] = ok
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else " — pas encore, relis l'énoncé ou ouvre l'indice."))

## Exercice 1 ⭐ · Classification ou régression ?

Pour chaque question, la machine doit-elle deviner une **catégorie** (classification, `"C"`) ou un **nombre**
(régression, `"R"`) ? Remplace les `"?"` du dictionnaire `reponses`.

Résultat attendu : 6 bonnes réponses sur 6.

<details><summary>Indice</summary>

Une catégorie se choisit dans une liste (un pays, une marque, oui / non) ; un nombre se mesure (des litres, des euros, des minutes).

</details>

In [ ]:
# À toi
reponses = {
    "conso_voiture": "?",            # deviner combien de litres aux 100 km consomme une voiture
    "pays_de_la_voiture": "?",       # deviner dans quel pays une voiture a été fabriquée
    "diamant_defaut_ou_pas": "?",    # deviner si un diamant a un défaut visible
    "prix_diamant": "?",             # deviner le prix d'un diamant
    "dejeuner_ou_diner": "?",        # deviner si une addition date du déjeuner ou du dîner
    "pourboire": "?",                # deviner le pourboire laissé sur une addition
}
print(reponses)

In [ ]:
corrige_1 = {"conso_voiture": "R", "pays_de_la_voiture": "C", "diamant_defaut_ou_pas": "C", "prix_diamant": "R", "dejeuner_ou_diner": "C", "pourboire": "R"}
score_1 = sum(reponses.get(k) == v for k, v in corrige_1.items())
print("Score :", score_1, "/ 6")
verifier("Exercice 1 · classification ou régression", score_1 == 6)

<details><summary>Solution</summary>

```python
reponses = {
    "conso_voiture": "R",
    "pays_de_la_voiture": "C",
    "diamant_defaut_ou_pas": "C",
    "prix_diamant": "R",
    "dejeuner_ou_diner": "C",
    "pourboire": "R",
}
print(reponses)
```

</details>

## Exercice 2 ⭐ · X et y

Prépare les données des voitures : `colonnes` (les 4 mesures `mpg`, `displacement`, `horsepower`, `weight`),
`X` (ce que la machine voit) et `y` (ce qu'elle doit deviner : `origin`). Compte `nb_origines`.

Résultat attendu : `X` a 392 lignes et 4 colonnes, `y` contient 3 origines.

<details><summary>Indice</summary>

`X = voitures[colonnes]` ; `y.nunique()` compte les valeurs différentes.

</details>

In [ ]:
# À toi
colonnes = []
X = None
y = None
nb_origines = None
print("X :", None if X is None else X.shape, "· origines :", nb_origines)

In [ ]:
verifier("Exercice 2 · X", isinstance(X, pd.DataFrame) and X.shape == (392, 4) and set(X.columns) == {"mpg", "displacement", "horsepower", "weight"})
verifier("Exercice 2 · y", y is not None and set(y.unique()) == {"usa", "japan", "europe"})
verifier("Exercice 2 · nombre d'origines", nb_origines == 3)

<details><summary>Solution</summary>

```python
colonnes = ["mpg", "displacement", "horsepower", "weight"]
X = voitures[colonnes]
y = voitures["origin"]
nb_origines = y.nunique()
print("X :", X.shape, "· origines :", nb_origines)
```

</details>

## Exercice 3 ⭐ · Cacher une partie des données

Coupe `X` et `y` en deux avec `train_test_split` : 25 % cachés pour le test, `random_state=42` pour que tout le monde
ait la même coupe. Calcule `part_test` (la part des données cachées, arrondie à 2 décimales).

Résultat attendu : 294 voitures pour apprendre, 98 cachées.

<details><summary>Indice</summary>

`X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)` ; `len(X_test) / len(X)`.

</details>

In [ ]:
# À toi
X_train, X_test, y_train, y_test = None, None, None, None
part_test = None
print("Apprentissage :", None if X_train is None else len(X_train), "· test :", None if X_test is None else len(X_test), "· part test :", part_test)

In [ ]:
verifier("Exercice 3 · la coupe", X_train is not None and len(X_train) == 294 and len(X_test) == 98 and len(y_test) == 98)
verifier("Exercice 3 · même coupe que tout le monde", X_test is not None and list(X_test.index[:3]) == [78, 274, 246])
verifier("Exercice 3 · part test", part_test == 0.25)

<details><summary>Solution</summary>

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
part_test = round(len(X_test) / len(X), 2)
print("Apprentissage :", len(X_train), "· test :", len(X_test), "· part test :", part_test)
```

</details>

## Exercice 4 ⭐ · Le modèle bête

Avant tout modèle, la barre à battre : un « modèle bête » qui répond toujours l'origine la plus fréquente du train.
Trouve `origine_frequente` et calcule `exactitude_bete` (la part de bonnes réponses sur les données de **test**).

Résultat attendu : un peu plus d'une bonne réponse sur deux, sans rien comprendre aux voitures.

<details><summary>Indice</summary>

`y_train.value_counts().index[0]` ; puis `(y_test == origine_frequente).mean()`.

</details>

In [ ]:
# À toi
origine_frequente = None
exactitude_bete = None
print("Le modèle bête répond toujours", origine_frequente, "→ exactitude :", exactitude_bete)

In [ ]:
verifier("Exercice 4 · origine la plus fréquente", origine_frequente == "usa")
verifier("Exercice 4 · exactitude du modèle bête", exactitude_bete is not None and round(float(exactitude_bete), 2) == 0.58)

<details><summary>Solution</summary>

```python
origine_frequente = y_train.value_counts().index[0]
exactitude_bete = (y_test == origine_frequente).mean()
print("Le modèle bête répond toujours", origine_frequente, "→ exactitude :", exactitude_bete)
```

</details>

## Exercice 5 ⭐⭐ · Ton premier arbre

Entraîne `arbre`, un `DecisionTreeClassifier` de profondeur 3 (`random_state=42`), fais-le prédire sur les voitures
cachées (`predictions`) et mesure `exactitude_arbre`. Puis trouve `premiere_question` : le nom de la colonne que l'arbre
utilise dans sa toute première question (à lire sur `plot_tree`, ou dans `arbre.tree_.feature[0]`). Affiche l'arbre.

Résultat attendu : mieux que le modèle bête, et une première question sur la cylindrée (les grosses cylindrées sont américaines).

<details><summary>Indice</summary>

`.fit(X_train, y_train)` puis `.predict(X_test)` ; `accuracy_score(y_test, predictions)` ; `colonnes[arbre.tree_.feature[0]]`.

</details>

In [ ]:
# À toi
arbre = None
predictions = None
exactitude_arbre = None
premiere_question = None
print("Exactitude de l'arbre :", exactitude_arbre, "· première question sur :", premiere_question)

In [ ]:
verifier("Exercice 5 · arbre entraîné", isinstance(arbre, DecisionTreeClassifier) and arbre.get_depth() == 3)
verifier("Exercice 5 · exactitude", exactitude_arbre is not None and round(float(exactitude_arbre), 3) == 0.694)
verifier("Exercice 5 · première question", premiere_question == "displacement")

<details><summary>Solution</summary>

```python
arbre = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
predictions = arbre.predict(X_test)
exactitude_arbre = accuracy_score(y_test, predictions)
premiere_question = colonnes[arbre.tree_.feature[0]]
print("Exactitude de l'arbre :", exactitude_arbre, "· première question sur :", premiere_question)
plt.figure(figsize=(16, 7))
plot_tree(arbre, feature_names=colonnes, class_names=arbre.classes_, filled=True, fontsize=9)
plt.show()
```

</details>

## Exercice 6 ⭐⭐ · La matrice de confusion

Où l'arbre se trompe-t-il ? Construis `matrice`, le tableau croisé « vraie origine × origine prédite » (`pd.crosstab`),
compte `nb_erreurs` (les cases hors de la diagonale) et trouve `origine_ratee` : l'origine que l'arbre rate le plus.

Résultat attendu : 30 erreurs sur 98, et une origine que l'arbre ne reconnaît presque jamais.

<details><summary>Indice</summary>

`pd.crosstab(y_test, predictions, rownames=["Vraie"], colnames=["Prédite"])` ; la diagonale : `np.trace(matrice.values)` ; `y_test != predictions` est vrai sur les erreurs, puis `y_test[erreurs].value_counts().idxmax()`.

</details>

In [ ]:
# À toi
matrice = None
nb_erreurs = None
origine_ratee = None
print(matrice)
print("Erreurs :", nb_erreurs, "· origine la plus ratée :", origine_ratee)

In [ ]:
verifier("Exercice 6 · matrice", matrice is not None and int(np.trace(np.asarray(matrice))) == 68)
verifier("Exercice 6 · erreurs", nb_erreurs == 30)
verifier("Exercice 6 · origine ratée", origine_ratee == "europe")

<details><summary>Solution</summary>

```python
matrice = pd.crosstab(y_test, predictions, rownames=["Vraie"], colnames=["Prédite"])
nb_erreurs = int(len(y_test) - np.trace(matrice.values))
erreurs = y_test != predictions
origine_ratee = y_test[erreurs].value_counts().idxmax()
print(matrice)
print("Erreurs :", nb_erreurs, "· origine la plus ratée :", origine_ratee)
```

Sur 24 voitures européennes cachées, l'arbre n'en reconnaît que 2 : il prend les autres pour des japonaises (13) ou des américaines (9). Avec ces quatre mesures, une petite européenne ressemble beaucoup à une petite japonaise.

</details>

## Exercice 7 ⭐⭐ · Les voisins et l'échelle

Entraîne un k-NN à 5 voisins et mesure `exactitude_knn`. Puis mets toutes les colonnes à la même échelle avec un
`StandardScaler` (ajusté sur le train, appliqué au train **et** au test), réentraîne et mesure `exactitude_knn_echelle`.
Conclus dans `meilleur` : `"avec"` ou `"sans"` mise à l'échelle ?

Résultat attendu : le poids (des milliers de livres) écrasait la consommation (des dizaines de miles) ; avec l'échelle, k-NN gagne 10 points.

<details><summary>Indice</summary>

`echelle = StandardScaler().fit(X_train)` puis `echelle.transform(X_train)` et `echelle.transform(X_test)`.

</details>

In [ ]:
# À toi
exactitude_knn = None
exactitude_knn_echelle = None
meilleur = "..."   # "avec" ou "sans"
print("k-NN sans échelle :", exactitude_knn, "· avec échelle :", exactitude_knn_echelle, "→ meilleur :", meilleur)

In [ ]:
verifier("Exercice 7 · k-NN brut", exactitude_knn is not None and round(float(exactitude_knn), 2) == 0.70)
verifier("Exercice 7 · k-NN mis à l'échelle", exactitude_knn_echelle is not None and round(float(exactitude_knn_echelle), 2) == 0.81)
verifier("Exercice 7 · conclusion", meilleur == "avec")

<details><summary>Solution</summary>

```python
voisins = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
exactitude_knn = accuracy_score(y_test, voisins.predict(X_test))
echelle = StandardScaler().fit(X_train)
voisins_echelle = KNeighborsClassifier(n_neighbors=5).fit(echelle.transform(X_train), y_train)
exactitude_knn_echelle = accuracy_score(y_test, voisins_echelle.predict(echelle.transform(X_test)))
meilleur = "avec"
print("k-NN sans échelle :", exactitude_knn, "· avec échelle :", exactitude_knn_echelle, "→ meilleur :", meilleur)
```

</details>

## Exercice 8 ⭐⭐ · Apprendre par cœur

Pour des profondeurs de 1 à 15, entraîne un arbre (`random_state=42`) et range dans `exact_train` et `exact_test`
son exactitude sur le train et sur le test. Trouve `profondeur_par_coeur` (la première profondeur où le train atteint
100 %) et `meilleure_profondeur_test` (la plus petite profondeur qui donne la meilleure exactitude de test).
Trace les deux courbes.

Résultat attendu : le train finit à 100 %, le test plafonne bien avant : c'est le sur-apprentissage.

<details><summary>Indice</summary>

`for p in range(1, 16):` ; `exact_train.index(1.0) + 1` ; `exact_test.index(max(exact_test)) + 1`.

</details>

In [ ]:
# À toi
profondeurs = list(range(1, 16))
exact_train, exact_test = [], []
profondeur_par_coeur = None
meilleure_profondeur_test = None
print("Par cœur à partir de la profondeur", profondeur_par_coeur, "· meilleure profondeur en test :", meilleure_profondeur_test)

In [ ]:
verifier("Exercice 8 · quinze arbres", len(exact_train) == 15 and len(exact_test) == 15 and exact_train[-1] == 1.0)
verifier("Exercice 8 · par cœur", profondeur_par_coeur == 9)
verifier("Exercice 8 · meilleure profondeur", meilleure_profondeur_test == 5)

<details><summary>Solution</summary>

```python
profondeurs = list(range(1, 16))
exact_train, exact_test = [], []
for p in profondeurs:
    a = DecisionTreeClassifier(max_depth=p, random_state=42).fit(X_train, y_train)
    exact_train.append(accuracy_score(y_train, a.predict(X_train)))
    exact_test.append(accuracy_score(y_test, a.predict(X_test)))
profondeur_par_coeur = exact_train.index(1.0) + 1
meilleure_profondeur_test = exact_test.index(max(exact_test)) + 1
print("Par cœur à partir de la profondeur", profondeur_par_coeur, "· meilleure profondeur en test :", meilleure_profondeur_test)
plt.figure(figsize=(8, 4))
plt.plot(profondeurs, exact_train, marker="o", label="train"); plt.plot(profondeurs, exact_test, marker="s", label="test")
plt.xlabel("Profondeur"); plt.ylabel("Exactitude"); plt.xticks(profondeurs); plt.legend()
plt.title("Le train monte à 100 %, le test plafonne dès la profondeur 5"); plt.show()
```

</details>

## Exercice 9 ⭐⭐ · Prédire un nombre

Changeons de famille : deviner la **consommation** (`mpg`) à partir de la cylindrée, des chevaux et du poids.
Coupe les données (`test_size=0.25`, `random_state=42`), entraîne une `LinearRegression` et mesure `erreur_regression`
(l'erreur absolue moyenne, en miles par gallon). Compare avec `erreur_bete` : l'erreur d'un modèle qui prédit toujours
la consommation moyenne du train.

Résultat attendu : la régression se trompe de 3,4 miles par gallon en moyenne, le modèle bête de près de 6.

<details><summary>Indice</summary>

`mean_absolute_error(yc_test, predictions)` ; pour le modèle bête, prédire `[yc_train.mean()] * len(yc_test)`.

</details>

In [ ]:
# À toi
colonnes_conso = ["displacement", "horsepower", "weight"]
erreur_regression = None
erreur_bete = None
print("Erreur de la régression :", erreur_regression, "· modèle bête :", erreur_bete)

In [ ]:
verifier("Exercice 9 · régression", erreur_regression is not None and abs(erreur_regression - 3.40) < 0.01)
verifier("Exercice 9 · modèle bête", erreur_bete is not None and abs(erreur_bete - 5.96) < 0.01)

<details><summary>Solution</summary>

```python
colonnes_conso = ["displacement", "horsepower", "weight"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(voitures[colonnes_conso], voitures["mpg"], test_size=0.25, random_state=42)
regression = LinearRegression().fit(Xc_train, yc_train)
erreur_regression = mean_absolute_error(yc_test, regression.predict(Xc_test))
erreur_bete = mean_absolute_error(yc_test, [yc_train.mean()] * len(yc_test))
print("Erreur de la régression :", erreur_regression, "· modèle bête :", erreur_bete)
```

</details>

## Exercice 10 ⭐⭐⭐ · Le piège des classes déséquilibrées

Sur les diamants, prédis si la pureté est la pire (`clarity == "I1"`) à partir de `carat`, `depth`, `table` et `price`.
Coupe avec `test_size=0.25`, `random_state=42` et `stratify=y_i1` (pour garder la même part de pierres I1 des deux côtés).
Calcule `exactitude_toujours_non` (un modèle qui répond « pas I1 » à tous les diamants), entraîne `arbre_i1`
(profondeur 3, `random_state=42`) et mesure `exactitude_arbre_i1`. Puis compte `i1_total_test` (les vraies pierres I1
du test) et `i1_retrouves` (celles que l'arbre a bien prédites).

Résultat attendu : le modèle bête et l'arbre ont la **même** exactitude, 99 %... et l'arbre ne retrouve presque aucune pierre I1.

<details><summary>Indice</summary>

`(y_i1_test == False).mean()` ; les retrouvées : `((pred_i1 == True) & (y_i1_test == True)).sum()`.

</details>

In [ ]:
# À toi
X_i1 = diamants[["carat", "depth", "table", "price"]]
y_i1 = diamants["clarity"] == "I1"
exactitude_toujours_non = None
arbre_i1 = None
exactitude_arbre_i1 = None
i1_total_test = None
i1_retrouves = None
print("Toujours non :", exactitude_toujours_non, "· arbre :", exactitude_arbre_i1)
print("Pierres I1 retrouvées :", i1_retrouves, "/", i1_total_test)

In [ ]:
verifier("Exercice 10 · toujours non", exactitude_toujours_non is not None and round(float(exactitude_toujours_non), 2) == 0.99)
verifier("Exercice 10 · arbre", exactitude_arbre_i1 is not None and round(float(exactitude_arbre_i1), 2) == 0.99)
verifier("Exercice 10 · pierres I1 du test", i1_total_test == 185)
verifier("Exercice 10 · pierres I1 retrouvées", i1_retrouves == 10)

<details><summary>Solution</summary>

```python
X_i1 = diamants[["carat", "depth", "table", "price"]]
y_i1 = diamants["clarity"] == "I1"
X_i1_train, X_i1_test, y_i1_train, y_i1_test = train_test_split(X_i1, y_i1, test_size=0.25, random_state=42, stratify=y_i1)
exactitude_toujours_non = (y_i1_test == False).mean()
arbre_i1 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_i1_train, y_i1_train)
pred_i1 = arbre_i1.predict(X_i1_test)
exactitude_arbre_i1 = accuracy_score(y_i1_test, pred_i1)
i1_total_test = int(y_i1_test.sum())
i1_retrouves = int(((pred_i1 == True) & (y_i1_test == True)).sum())
print("Toujours non :", exactitude_toujours_non, "· arbre :", exactitude_arbre_i1)
print("Pierres I1 retrouvées :", i1_retrouves, "/", i1_total_test)
print(pd.crosstab(y_i1_test, pred_i1, rownames=["Vrai"], colnames=["Prédit"]))
```

</details>

## Exercice 11 ⭐⭐⭐ · Mesurer ce qui compte : le rappel

Quand une classe est rare, on regarde le **rappel** : la part des vraies pierres I1 que le modèle retrouve
(`retrouvées / total`). Calcule `rappel_3` pour l'arbre de profondeur 3, puis entraîne un arbre de profondeur 8 et
calcule `exactitude_8` et `rappel_8`. Un bijoutier veut **trouver** les pierres I1 avant de les vendre : quelle profondeur
choisis-tu ? Réponds dans `meilleure_profondeur`.

Résultat attendu : l'exactitude bouge à peine, le rappel est multiplié par six.

<details><summary>Indice</summary>

`rappel_3 = i1_retrouves / i1_total_test` ; pour la profondeur 8, les mêmes calculs qu'à l'exercice 10 avec `max_depth=8`.

</details>

In [ ]:
# À toi
rappel_3 = None
exactitude_8 = None
rappel_8 = None
meilleure_profondeur = None
print("Profondeur 3 → rappel", rappel_3, "· profondeur 8 → exactitude", exactitude_8, ", rappel", rappel_8)
print("Je choisis la profondeur", meilleure_profondeur)

In [ ]:
verifier("Exercice 11 · rappel profondeur 3", rappel_3 is not None and round(float(rappel_3), 2) == 0.05)
verifier("Exercice 11 · exactitude profondeur 8", exactitude_8 is not None and round(float(exactitude_8), 3) == 0.989)
verifier("Exercice 11 · rappel profondeur 8", rappel_8 is not None and round(float(rappel_8), 2) == 0.32)
verifier("Exercice 11 · choix", meilleure_profondeur == 8)

<details><summary>Solution</summary>

```python
rappel_3 = i1_retrouves / i1_total_test
arbre_8 = DecisionTreeClassifier(max_depth=8, random_state=42).fit(X_i1_train, y_i1_train)
pred_8 = arbre_8.predict(X_i1_test)
exactitude_8 = accuracy_score(y_i1_test, pred_8)
rappel_8 = int(((pred_8 == True) & (y_i1_test == True)).sum()) / i1_total_test
meilleure_profondeur = 8
print("Profondeur 3 → rappel", rappel_3, "· profondeur 8 → exactitude", exactitude_8, ", rappel", rappel_8)
print("Je choisis la profondeur", meilleure_profondeur)   # 98,6 % → 98,9 % d'exactitude, mais 10 → 60 pierres I1 retrouvées sur 185
```

</details>

## Exercice 12 ⭐⭐⭐ · Défi : la recette complète sur les additions

Le montant, le pourboire et le nombre de convives suffisent-ils à deviner si une addition date du **déjeuner** ou du
**dîner** ? Prends `total_bill`, `tip` et `size` comme `X_add` et `time` comme `y_add`, coupe (`test_size=0.25`,
`random_state=42`, sans `stratify`), puis remplis `resultats_add` avec trois exactitudes : `"bête"` (le moment le plus
fréquent du train), `"arbre"` (profondeur 3, `random_state=42`) et `"k-NN"` (5 voisins, avec `StandardScaler`).
Conclus : `donnees_suffisent` (`True` / `False`) et une `phrase` qui explique ce que ton modèle a compris... ou pas.

Résultat attendu : aucun modèle ne bat le modèle bête. Un bon modèle ne peut pas deviner ce qui n'est pas dans les données !

<details><summary>Indice</summary>

Le reste est la recette des exercices 2 à 7 : X et y, la coupe, le modèle bête, l'arbre, k-NN avec l'échelle.

</details>

In [ ]:
# À toi
X_add = None
y_add = None
resultats_add = {"bête": None, "arbre": None, "k-NN": None}
donnees_suffisent = None
phrase = "Mon modèle a compris que ..."
print(resultats_add)
print(phrase)

In [ ]:
verifier("Défi · X et y", isinstance(X_add, pd.DataFrame) and X_add.shape == (244, 3) and y_add is not None and set(y_add.unique()) == {"Lunch", "Dinner"})
verifier("Défi · modèle bête", resultats_add.get("bête") is not None and round(float(resultats_add["bête"]), 2) == 0.72)
verifier("Défi · arbre et k-NN mesurés", all(resultats_add.get(k) is not None and 0 <= float(resultats_add[k]) < float(resultats_add["bête"]) for k in ["arbre", "k-NN"]))
verifier("Défi · conclusion", donnees_suffisent is False and len(phrase) > 30 and "..." not in phrase)

<details><summary>Solution</summary>

```python
X_add = additions[["total_bill", "tip", "size"]]
y_add = additions["time"]
Xa_train, Xa_test, ya_train, ya_test = train_test_split(X_add, y_add, test_size=0.25, random_state=42)

resultats_add = {}
resultats_add["bête"] = (ya_test == ya_train.value_counts().index[0]).mean()
arbre_add = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xa_train, ya_train)
resultats_add["arbre"] = accuracy_score(ya_test, arbre_add.predict(Xa_test))
echelle_add = StandardScaler().fit(Xa_train)
knn_add = KNeighborsClassifier(n_neighbors=5).fit(echelle_add.transform(Xa_train), ya_train)
resultats_add["k-NN"] = accuracy_score(ya_test, knn_add.predict(echelle_add.transform(Xa_test)))

donnees_suffisent = False
phrase = "Mon modèle n'a rien compris d'utile : avec le montant, le pourboire et le nombre de convives, ni l'arbre ni k-NN ne font mieux que répondre toujours Dinner."
print(resultats_add)
print(phrase)
```

Bête 72 %, arbre 67 %, k-NN 64 % : les deux modèles font **moins bien** que le modèle bête. Un déjeuner et un dîner peuvent coûter la même chose : l'information n'est pas dans ces trois colonnes.

</details>

## Bravo !

Tu as fait le tour du premier modèle sur trois jeux de données nouveaux : X et y, train / test, le modèle bête, l'arbre
qu'on peut lire, k-NN et l'échelle, le sur-apprentissage, une régression, le piège des classes déséquilibrées (regarde le
rappel, pas seulement l'exactitude), et un cas où aucun modèle ne bat le modèle bête. C'est exactement la recette que tu
vas appliquer sur le Titanic aux séances 10 et 11.

## Bilan

La cellule ci-dessous compte tes ✅. Relance-la après chaque exercice réussi.

In [ ]:
reussis = sum(1 for ok in resultats.values() if ok)
print(f"{reussis} exercice(s) réussi(s) sur {len(resultats)}")
for nom, ok in resultats.items():
    print("  ", "✅" if ok else "❌", nom)